# Tech Challenge: Classificando a qualidade de vinhos com Machine Learning

Este notebook atende ao desafio de prever a qualidade de
um vinho com base em suas características físico-químicas.
- Quality >= 7: Vinho de alta qualidade.
- Quality < 7: Vinho de baixa/média qualidade.

## 1. Compreensão do Problema

- Variável alvo original: quality
- Transformação binária: quality_binary = 1 se quality >= 7, senão 0
- Objetivo: Treinar e avaliar modelos de aprendizado de máquina capazes de prever essa classificação a partir das variáveis disponíveis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    )

# Configura estilo dos gráficos para melhorar legibilidade.
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

In [ ]:
# Localiza a raiz do projeto tanto ao abrir pela raiz quanto pela pasta notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / 'data' / 'winequality-red.csv').exists():
    project_root = project_root.parent

data_path = project_root / 'data' / 'winequality-red.csv'
results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

if not data_path.exists():
    raise FileNotFoundError(f'Dataset não encontrado em: {data_path}')

# Leitura do dataset de vinhos.
df = pd.read_csv(data_path)

# Transforma a qualidade original em alvo binário:
# 1 = alta qualidade (quality >= 7), 0 = baixa/media (quality < 7).
df['quality_binary'] = (df['quality'] >= 7).astype(int)

df.head()

In [ ]:
#Números representando  a dimensionalidade do conjunto de dados
df.shape

## 2. Análise Exploratória de Dados (EDA)



In [ ]:
# Visão geral da base.
print('Dimensão da base (linhas, colunas):', df.shape)
print('\nQuantidade de valores nulos por coluna:')
print(df.isnull().sum())

# Gráfico 1: balanceamento das classes.
# Ele mostra quantos vinhos há em cada classe do alvo binário.
plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x='quality_binary', hue='quality_binary', palette='Set2', legend=False)
ax.set_title('Balanceamento das Classes', fontsize=13, weight='bold')
ax.set_xlabel('Classe alvo (0 = baixa/media, 1 = alta)')
ax.set_ylabel('Quantidade de amostras')

# Adiciona os valores no topo de cada barra para leitura imediata.
for p in ax.patches:
    altura = int(p.get_height())
    ax.annotate(
        f'{altura}',
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha='center',
        va='bottom',
        fontsize=10,
        xytext=(0, 3),
        textcoords='offset points'
    )

plt.tight_layout()
plt.savefig(results_dir / 'balanceamento_classes.png', dpi=300, bbox_inches='tight')
plt.show()

proporcao = df['quality_binary'].value_counts(normalize=True).sort_index()
print('\nInterpretação do gráfico:')
print(f"- Classe 0 (baixa/media): {proporcao.loc[0]*100:.1f}%")
print(f"- Classe 1 (alta): {proporcao.loc[1]*100:.1f}%")
print('- Quanto mais desbalanceadas as barras, maior o cuidado na avaliação dos modelos.')

In [ ]:
# Gráfico 2: mapa de calor de correlações entre variáveis numéricas.
# Cores próximas de vermelho = correlação positiva forte.
# Cores próximas de azul = correlação negativa forte.
corr = df.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
ax = sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    linewidths=0.3,
    cbar_kws={'label': 'Coeficiente de correlação'}
)
ax.set_title('Matriz de Correlação das Variáveis', fontsize=13, weight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(results_dir / 'matriz_correlacao.png', dpi=300, bbox_inches='tight')
plt.show()

# Mostra automaticamente as correlações mais fortes com o alvo para facilitar interpretação.
corr_alvo = corr['quality_binary'].drop('quality_binary').sort_values(key=np.abs, ascending=False)
print('Interpretação do gráfico (top 6 correlações com qualidade_binary):')
for nome, valor in corr_alvo.head(6).items():
    direcao = 'positiva' if valor >= 0 else 'negativa'
    print(f'- {nome}: correlação {direcao} ({valor:.3f})')

print('\nLeitura rápida: valores mais próximos de +1 ou -1 indicam relação mais forte com a classe alvo.')

## 3. Preparação dos Dados para Machine Learning

Nesta etapa, a base será preparada para o treinamento dos modelos de classificação. 
A variável `quality_binary` será utilizada como alvo, onde:

- 1 representa vinho de alta qualidade (`quality >= 7`);
- 0 representa vinho de baixa/média qualidade (`quality < 7`).

A coluna `quality` será removida das variáveis de entrada para evitar vazamento de informação, pois ela foi usada para criar a variável alvo binária.

In [ ]:
# A base já foi carregada e a variável quality_binary foi criada nas etapas anteriores.
# Reutilizar o mesmo DataFrame evita depender de um segundo caminho para o CSV.

# Visualização inicial
display(df.head())

# Conferência das colunas
print("Colunas da base:")
print(df.columns.tolist())

# Separação das variáveis explicativas e da variável alvo
X = df.drop(columns=["quality", "quality_binary"], errors="ignore")
y = df["quality_binary"]

print("\nColunas usadas no modelo:")
print(X.columns.tolist())

print("\nDistribuição da variável alvo:")
print(y.value_counts())

print("\nProporção da variável alvo:")
print(y.value_counts(normalize=True))